# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR<sup>2</sup> dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore the available record sets and their fields. All references are via `@id`.

We'll inspect all record sets and list their `@id`, `name`, and included fields.

In [ ]:
# List all record sets with their @id, name, and fields
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in the dataset schema. Trying to extract from underlying distributions...")
    # If the Dataset metadata does not enumerate record_sets, infer from the dataset API
    print("Available record sets as found by mlcroissant:")
    for rs in dataset.available_record_sets():
        print(f"- @id: {rs.get('@id', 'N/A')}, name: {rs.get('name', 'N/A')}")
else:
    for rs in record_sets:
        print(f"- @id: {rs['@id']}, name: {getattr(rs, 'name', 'N/A')}")
        if hasattr(rs, 'fields'):
            fields = rs.fields
            if isinstance(fields, list):
                print("  Fields:")
                for field in fields:
                    print(f"    - @id: {field['@id']}, name: {getattr(field, 'name', 'N/A')}")
            else:
                print("  No fields found.")
        else:
            print("  No fields found.")

## 3. Data Extraction
Identify one or more record sets (using their `@id`) and load them into Pandas DataFrames for analysis.

For this dataset, the primary data appears to be in a single CSV file named with a record set id like `'https://sen.science/doi/10.71728/senscience.qs2f-h81p/recordset/0'`.

We'll show how to load **all** available record sets (if more than one), referencing them with their `@id`s.

In [ ]:
# Discover available record set @id(s)
record_set_ids = dataset.available_record_set_ids()
print("Record set @id(s) detected:")
for rsid in record_set_ids:
    print(f"- {rsid}")

# Load data from each record set into a DataFrame
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)
    print(f"Loaded record set {rsid} with shape {dataframes[rsid].shape}")

# Show columns in the first record set
first_rsid = record_set_ids[0]
print("\nColumns in first record set:")
print(dataframes[first_rsid].columns.tolist())
dataframes[first_rsid].head()

## 4. Exploratory Data Analysis (EDA)

We'll now explore the data, filter by a numeric field, normalize it, and group by a chosen categorical/group field. All field selections should be done by referencing the columns using their `@id` as detected earlier.

In [ ]:
# Select record set @id to analyze (primary table)
record_set_id = first_rsid
df = dataframes[record_set_id]

# List all columns to help select a numeric field and group field
print("Columns (field @id):")
for i, col in enumerate(df.columns):
    print(f"{i}: {col}")

# Example: Let's pick a likely numeric field (for demo: let's try columns with int or float type)
numeric_col = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        if df[col].notnull().any():
            numeric_col = col
            break
if numeric_col is None:
    # Fallback: try to coerce one column to numeric
    for col in df.columns:
        try:
            asnum = pd.to_numeric(df[col], errors='coerce')
            if asnum.notnull().any():
                numeric_col = col
                df[col] = asnum
                break
        except Exception:
            continue

if numeric_col is None:
    raise ValueError("Could not find a numeric field in the dataset.")

print(f"\nNumeric field chosen (field @id): {numeric_col}")
print(f"Summary statistics for {numeric_col}:")
print(df[numeric_col].describe())

# Pick a threshold that's 1 std above mean (as in template; here we compute a data-driven threshold)
threshold = df[numeric_col].mean() + df[numeric_col].std()
filtered_df = df[df[numeric_col] > threshold]
print(f"\nFiltered records with {numeric_col} > {threshold:.2f} (n={len(filtered_df)}):")
print(filtered_df[[numeric_col]].head(3))

# Normalize the numeric field
filtered_df.loc[:, f"{numeric_col}_normalized"] = (
    (filtered_df[numeric_col] - df[numeric_col].mean()) / df[numeric_col].std()
)
print(f"\nNormalized field shown for filtered records:")
print(filtered_df[[numeric_col, f"{numeric_col}_normalized"].copy()].head(3))

# Try to pick a group/categorical field for grouping/aggregation
group_field = None
for col in df.columns:
    # Pick the first non-numeric, non-null field as group
    if not pd.api.types.is_numeric_dtype(df[col]):
        if df[col].nunique() > 1 and df[col].notnull().any():
            group_field = col
            break

if group_field:
    print(f"\nGrouping filtered records by field @id: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_col].mean()
    print(grouped_df.head())
else:
    print("\nNo suitable group field detected for aggregation.")

## 5. Visualization
Visualize the distribution and relationships of selected fields using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the chosen numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_col].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_col}")
plt.xlabel(numeric_col)
plt.ylabel("Count")
plt.show()

# Box plot by group field if available
if group_field:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field, y=numeric_col, data=df)
    plt.title(f"{numeric_col} by {group_field}")
    plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded the FAIR<sup>2</sup> colorectal cancer dataset using its Croissant schema (@id referenced throughout).
- Explored available record sets and fields.
- Extracted the primary data table into a Pandas DataFrame by referencing its `@id`.
- Performed initial EDA: filtered, normalized, and grouped data by key attributes.
- Visualized distributions and group-wise means.

This template demonstrates how to use `mlcroissant` for reproducible, schema-driven tabular data exploration.

**You can now adapt this notebook to your analysis questions, referencing fields and record sets directly by their `@id`.**